# 04 - Carga a MongoDB Atlas
Objetivo: transformar CSV relacionales a documentos optimizados.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from pymongo import MongoClient
from dotenv import load_dotenv
load_dotenv()
client = MongoClient(os.environ['MONGODB_URI'])
db = client[os.getenv('MONGODB_DATABASE', 'ecommify')]
DATA_DIR = Path(os.getenv('DATA_DIR', '/content/data'))

In [ ]:
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
cats = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')
products = products.merge(cats, on='product_category_name', how='left')

def product_doc(r):
    return {
        'product_id': r['product_id'],
        'category': {'name_pt': r.get('product_category_name'), 'name_en': r.get('product_category_name_english')},
        'metrics': {
            'name_length': None if pd.isna(r.get('product_name_lenght')) else int(r.get('product_name_lenght')),
            'description_length': None if pd.isna(r.get('product_description_lenght')) else int(r.get('product_description_lenght')),
            'photos_qty': None if pd.isna(r.get('product_photos_qty')) else int(r.get('product_photos_qty'))
        },
        'attributes': [
            {'k': 'weight_g', 'v': None if pd.isna(r.get('product_weight_g')) else float(r.get('product_weight_g'))},
            {'k': 'length_cm', 'v': None if pd.isna(r.get('product_length_cm')) else float(r.get('product_length_cm'))},
            {'k': 'height_cm', 'v': None if pd.isna(r.get('product_height_cm')) else float(r.get('product_height_cm'))},
            {'k': 'width_cm', 'v': None if pd.isna(r.get('product_width_cm')) else float(r.get('product_width_cm'))}
        ],
        'dimensions': {
            'weight_g': None if pd.isna(r.get('product_weight_g')) else float(r.get('product_weight_g')),
            'length_cm': None if pd.isna(r.get('product_length_cm')) else float(r.get('product_length_cm')),
            'height_cm': None if pd.isna(r.get('product_height_cm')) else float(r.get('product_height_cm')),
            'width_cm': None if pd.isna(r.get('product_width_cm')) else float(r.get('product_width_cm')),
            'volume_cm3': None if pd.isna(r.get('product_length_cm')) or pd.isna(r.get('product_height_cm')) or pd.isna(r.get('product_width_cm')) else float(r.get('product_length_cm') * r.get('product_height_cm') * r.get('product_width_cm'))
        }
    }

docs = [product_doc(r) for _, r in products.iterrows()]
db.products_catalog.delete_many({})
db.products_catalog.insert_many(docs, ordered=False)
print('products_catalog', db.products_catalog.count_documents({}))

In [ ]:
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp'])
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
pay_summary = payments.groupby('order_id').agg(
    total_value=('payment_value', 'sum'),
    payment_types=('payment_type', lambda s: list(sorted(set(s.astype(str))))),
    installments_max=('payment_installments', 'max')
).reset_index()
orders = orders.merge(customers, on='customer_id', how='left').merge(pay_summary, on='order_id', how='left')

def order_doc(r):
    ts = r['order_purchase_timestamp'].to_pydatetime() if not pd.isna(r['order_purchase_timestamp']) else None
    return {
        'order_id': r['order_id'],
        'status': r['order_status'],
        'purchase_ts': ts,
        'purchase_year_month': None if ts is None else ts.strftime('%Y-%m'),
        'customer': {
            'customer_id': r['customer_id'],
            'customer_unique_id': r.get('customer_unique_id'),
            'state': r.get('customer_state'),
            'city': r.get('customer_city'),
            'zip_code_prefix': None if pd.isna(r.get('customer_zip_code_prefix')) else int(r.get('customer_zip_code_prefix'))
        },
        'payment_summary': {
            'total_value': 0 if pd.isna(r.get('total_value')) else float(r.get('total_value')),
            'payment_types': [] if isinstance(r.get('payment_types'), float) else r.get('payment_types'),
            'installments_max': None if pd.isna(r.get('installments_max')) else int(r.get('installments_max'))
        }
    }

docs = [order_doc(r) for _, r in orders.iterrows()]
db.orders_analytics.delete_many({})
db.orders_analytics.insert_many(docs, ordered=False)
print('orders_analytics', db.orders_analytics.count_documents({}))